# L4: Tools for a Customer Outreach Campaign

In this lesson, you will learn more about Tools. You'll focus on three key elements of Tools:
- Versatility
- Fault Tolerance
- Caching

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM
- [Serper](https://serper.dev)

In [3]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [1]:
import os
from com.example.agentic.utils import get_openai_api_key, get_serper_api_key, pretty_print_result
from com.example.agentic.utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'llama3.2:1b-instruct-q8_0'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## Creating Agents

In [4]:
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match "
         "our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at CrewAI, "
        "your mission is to scour "
        "the digital landscape for potential leads. "
        "Armed with cutting-edge tools "
        "and a strategic mindset, you analyze data, "
        "trends, and interactions to "
        "unearth opportunities that others might overlook. "
        "Your work is crucial in paving the way "
        "for meaningful engagements and driving the company's growth."
    ),
    allow_delegation=False,
    verbose=True
)

In [5]:
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within the vibrant ecosystem of CrewAI's sales department, "
        "you stand out as the bridge between potential clients "
        "and the solutions they need."
        "By creating engaging, personalized messages, "
        "you not only inform leads about our offerings "
        "but also make them feel seen and heard."
        "Your role is pivotal in converting interest "
        "into action, guiding leads through the journey "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
    verbose=True
)

## Creating Tools

### crewAI Tools

In [6]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [8]:
directory_read_tool = DirectoryReadTool(directory='/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

### Custom Tool
- Create a custom tool using crewAi's [BaseTool](https://docs.crewai.com/core-concepts/Tools/#subclassing-basetool) class

In [9]:
from crewai.tools import BaseTool

- Every Tool needs to have a `name` and a `description`.
- For simplicity and classroom purposes, `SentimentAnalysisTool` will return `positive` for every text.
- When running locally, you can customize the code with your logic in the `_run` function.

In [10]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [11]:
sentiment_analysis_tool = SentimentAnalysisTool()

## Creating Tasks

- The Lead Profiling Task is using crewAI Tools.

In [12]:
lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector "
        "that recently showed interest in our solutions. "
        "Utilize all available data sources "
        "to compile a detailed profile, "
        "focusing on key decision-makers, recent business "
        "developments, and potential needs "
        "that align with our offerings. "
        "This task is crucial for tailoring "
        "our engagement strategy effectively.\n"
        "Don't make assumptions and "
        "only use information you absolutely sure about."
    ),
    expected_output=(
        "A comprehensive report on {lead_name}, "
        "including company background, "
        "key personnel, recent milestones, and identified needs. "
        "Highlight potential areas where "
        "our solutions can provide value, "
        "and suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

- The Personalized Outreach Task is using your custom Tool `SentimentAnalysisTool`, as well as crewAI's `SerperDevTool` (search_tool).

In [13]:
personalized_outreach_task = Task(
    description=(
        "Using the insights gathered from "
        "the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign "
        "aimed at {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "The campaign should address their recent {milestone} "
        "and how our solutions can support their goals. "
        "Your communication must resonate "
        "with {lead_name}'s company culture and values, "
        "demonstrating a deep understanding of "
        "their business and needs.\n"
        "Don't make assumptions and only "
        "use information you absolutely sure about."
    ),
    expected_output=(
        "A series of personalized email drafts "
        "tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}."
        "Each draft should include "
        "a compelling narrative that connects our solutions "
        "with their recent achievements and future goals. "
        "Ensure the tone is engaging, professional, "
        "and aligned with {lead_name}'s corporate identity."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

## Creating the Crew

In [ ]:
from crewai import LLM, Memory
from crewai.rag.embeddings.factory import build_embedder, build_embedder_from_provider

_embedder = build_embedder({ 
    "provider": "openai", 
    "config": {"model_name":"nomic-embed-text:latest"}
})

memory = Memory(
    llm=LLM(model="openai/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434/v1"),
    embedder=_embedder
)

crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=True,
	memory=memory
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [16]:
inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Online Learning Platform",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "product launch"
}

result = crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 75ae8b9a-2160-4971-a20b-f3a8b29a488e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Invalid type Memory for attribute 'crew_memory' value. Expected one of ['bool', 'str', 'bytes', 'int', 'float'] or a sequence of those types


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Conduct an in-depth analysis of DeepLearningAI, a company in the Online Learning Platform sector that    │
│  recently showed interest in our solutions. Utilize all available data sources to compile a detailed profile,   │
│  focusing on key decision-makers, recent business developments, and potential needs that align with our         │
│  offerings. This task is crucial for tailoring our engagement strategy effectively.                             │
│  Don't make assumptions and only use information you absolutely sure about.                                     │
│  ID: 1d173142-b3bd-4a54-9222-2481f426f5c1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Query analysis failed, using defaults (complexity=simple): Expecting value: line 1 column 1 (char 0)


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the insights gathered from the lead profiling report on DeepLearningAI, craft a personalized       │
│  outreach campaign aimed at Andrew Ng, the CEO of DeepLearningAI. The campaign should address their recent      │
│  product launch and how our solutions can support their goals. Your communication must resonate with            │
│  DeepLearningAI's company culture and values, demonstrating a deep understanding of their business and needs.   │
│  Don't make assumptions and only use information you absolutely sure about.                                     │
│  Agent: Lead Sales Representative                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 75ae8b9a-2160-4971-a20b-f3a8b29a488e                                                                       │
│  Final Output: {                                                                                                │
│                                                                                                                 │
│    "name": "analyze DeepLearningAI",                                                                            │
│    "parameters": {                                                                                              │
│      "query": "DeepLearningAI Company Profile",                                                                 │
│      "start_line": null,                                                                                        │
│      "line_count": null                                                                                         │
│    },                                                                                                           │
│    "entity": "Andrew Ng",                                                                                       │
│    "message_type": "personalized_email",                                                                        │
│    "subject": "Your Path Forward with CrewAI: Aligning Our Solutions with DeepLearningAI's Goals",              │
│    "body": {                                                                                                    │
│      "Dear Andrew Ng",                                                                                          │
│      "Thank you for taking the time to review our profile.\n\nAccording to our analysis, your recent product    │
│  launch demonstrates an increasing interest in advanced AI solutions.\nCrewAI is committed to providing         │
│  innovative tools like ours that support your goals and objectives.\n\nHere are some ways CrewAI's solutions    │
│  can benefit DeepLearningAI:\n- Streamline Your Operations: Our cutting-edge AI-powered solutions can help      │
│  automate tasks, enhance decision-making, and boost efficiency in your operations.\n- Stay Ahead of the         │
│  Competition: By leveraging our solutions, you'll be able to gain a competitive edge by staying ahead of        │
│  emerging trends and technologies.\n- Improve Customer Experience: Our AI-driven features can lead to enhanced  │
│  customer engagement, better experience, and increased loyalty.\n\nIn particular, we'd like to highlight how    │
│  our solutions can support your:\n—\n* Data Science Workload\n—\n* Research and Development\n—\n* Marketing     │
│  Efforts\nWe believe that our collaboration could have positive impacts on each other's projects. We'd be       │
│  delighted to schedule a follow-up call to discuss the details further.\n\nBest regards.",                      │
│      "personalized": true,                                                                                      │
│      "tone": "engaging",                                                                                        │
│      "customer_type": "independent"                                                                             │
│    }                                                                                                            │
│  }                                                                                                              │
│                                                       

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 3045e3d7-d1ac-42b1-87f5-432bd581068b                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/3045e3d7-d1ac-42b1-87f5-432bd581068b?access_code=TRA │
│ CE-a85320904d                                                                                                   │
│ 🔑 Access Code: TRACE-a85320904d                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 38185.19ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [18]:
from IPython.display import Markdown
Markdown(result.raw)

{

  "name": "analyze DeepLearningAI",
  "parameters": {
    "query": "DeepLearningAI Company Profile",
    "start_line": null,
    "line_count": null
  },
  "entity": "Andrew Ng",
  "message_type": "personalized_email",
  "subject": "Your Path Forward with CrewAI: Aligning Our Solutions with DeepLearningAI's Goals",
  "body": {
    "Dear Andrew Ng",
    "Thank you for taking the time to review our profile.\n\nAccording to our analysis, your recent product launch demonstrates an increasing interest in advanced AI solutions.\nCrewAI is committed to providing innovative tools like ours that support your goals and objectives.\n\nHere are some ways CrewAI's solutions can benefit DeepLearningAI:\n- Streamline Your Operations: Our cutting-edge AI-powered solutions can help automate tasks, enhance decision-making, and boost efficiency in your operations.\n- Stay Ahead of the Competition: By leveraging our solutions, you'll be able to gain a competitive edge by staying ahead of emerging trends and technologies.\n- Improve Customer Experience: Our AI-driven features can lead to enhanced customer engagement, better experience, and increased loyalty.\n\nIn particular, we'd like to highlight how our solutions can support your:\n—\n* Data Science Workload\n—\n* Research and Development\n—\n* Marketing Efforts\nWe believe that our collaboration could have positive impacts on each other's projects. We'd be delighted to schedule a follow-up call to discuss the details further.\n\nBest regards.",
    "personalized": true,
    "tone": "engaging",
    "customer_type": "independent"
  }
}